In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.optim import Adam
from torchvision import datasets, transforms
from SupConLosszy import SupConLoss
import copy


In [2]:
print('==> Preparing............................')
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder


import os
import torch
from torch import nn,optim
import torch.nn.functional as F
from torchvision import datasets, transforms

from time import perf_counter

import  numpy as np
import torch.utils.data as Data

from torch.utils.data import Dataset, DataLoader

==> Preparing............................


In [3]:
print('==> Preparing............................')
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder


import os
import torch
from torch import nn,optim
import torch.nn.functional as F
from torchvision import datasets, transforms

from time import perf_counter

import  numpy as np
import torch.utils.data as Data

from torch.utils.data import Dataset, DataLoader



class Safeman(Dataset): 
    
    def __init__(self, data,targets):  #- **`__init__`**: 初始化函数，接收数据和标签，并将它们存储在实例变量 `self.data` 和 `self.targets` 中。
        super(Safeman, self).__init__()
        self.data = data
        self.targets = targets
        
     
    def __len__(self): #__len__: 返回数据集的大小，即标签的数量。
        return len(self.targets)

    def __getitem__(self, idx): #__getitem__: 根据索引 idx 返回对应的数据和标签。
        img, target = self.data[idx], self.targets[idx]
        return img, target

class Safeman_Filter(Safeman):

    def __Filter__(self, known): #- **`__Filter__`**: 过滤函数，接收一个已知类别列表 `known`。
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)
        
class Safeman_FilterB(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        new_targets = []
        for i in range(len(targets)):
            if targets[i] in known:
                new_targets.append(0)
            else:
                new_targets.append(1)
        self.targets = np.array(new_targets)
        self.data = self.data

class Safeman_FilterC(Safeman):
    
    def __Filter__(self, trainknown):
        train_class_num=len(trainknown)
        for i in range(0,len(self.targets)) :
            if self.targets[i]>train_class_num:
                self.targets[i] = train_class_num
        self.data = self.data

        
def setup_seed(seed):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(8)



known=[0, 1, 2,3]
unknown=[4]

num_class=len(known)


X_train0 = np.load('C:/Users/lenovo/Desktop/我的论文/论文4/数据集/X_train.npy')
y_train1 = np.load('C:/Users/lenovo/Desktop/我的论文/论文4/数据集/y_train.npy')
X_final_test0 = np.load('C:/Users/lenovo/Desktop/我的论文/论文4/数据集/X_test.npy' )
y__final_test1 = np.load('C:/Users/lenovo/Desktop/我的论文/论文4/数据集/y_test.npy')

X_train1=[]
X_final_test1=[]

for i in range(len(y_train1)):
    a = np.resize(X_train0[i], (1, 28, 28))
    X_train1 += [a]
    
for j in range(len(y__final_test1)):
    b = np.resize(X_final_test0[j], (1, 28, 28))
    X_final_test1 += [b]

i=0
j=0



x_train, x_test, y_train,y_test = torch.Tensor(X_train1), torch.Tensor(X_final_test1), torch.from_numpy(y_train1), torch.from_numpy(y__final_test1)

print(x_train.shape, x_test.shape, y_train.shape,y_test.shape)

train_dataset = Data.TensorDataset(x_train, y_train)
train_dataset.data = train_dataset.tensors[0]
train_dataset.targets = train_dataset.tensors[1]



test_dataset = Data.TensorDataset(x_test, y_test)
test_dataset.data = test_dataset.tensors[0]
test_dataset.targets = test_dataset.tensors[1]


labels =['ARP poisining', 'BeEF HTTP exploits','Mass HTTP requests','Metasploit exploits','Normal','Port scanning','TCP flood','UDP data flood']



train_dataset.classes = labels
test_dataset.classes = labels

train_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}
test_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}



b_s=16

trainset = Safeman_Filter(data=train_dataset.data,targets=train_dataset.targets)
print('All down Train Data:', len(trainset))
trainset.__Filter__(known=known)


train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=b_s, shuffle=True,
    num_workers=0,drop_last=True)


print('Real train Data:', len(trainset))



testsetA = Safeman_Filter(data=test_dataset.data,targets=test_dataset.targets)
print('All testsetA Data:', len(testsetA))
testsetA.__Filter__(known=known)


test_loader_A = torch.utils.data.DataLoader(
    testsetA, batch_size=b_s, shuffle=True,
    num_workers=0,drop_last=True)


print('Real testsetA Data:', len(testsetA))


print("done!")

==> Preparing............................


C:\Users\lenovo\AppData\Local\Temp\ipykernel_1632\2528703838.py:122: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at  C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_new.cpp:204.)
  x_train, x_test, y_train,y_test = torch.Tensor(X_train1), torch.Tensor(X_final_test1), torch.from_numpy(y_train1), torch.from_numpy(y__final_test1)


torch.Size([654751, 1, 28, 28]) torch.Size([163688, 1, 28, 28]) torch.Size([654751]) torch.Size([163688])
All down Train Data: 654751
Real train Data: 635111
All testsetA Data: 163688
Real testsetA Data: 158704
done!


In [4]:



from torch.nn import Module
from torch import nn
import numpy as np
import torch
from torchvision.datasets import mnist
from torch.nn import CrossEntropyLoss
from torch.optim import SGD
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor

class Modelnsl8(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 64, 32)
        self.fc2 = nn.Linear(32, num_class)
      
    def forward(self, x):
        x = x.view(x.size(0), 1, -1)  # Reshape to (batch_size, 1, 784)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = F.relu(self.fc1(x))
        feat = F.normalize(x, dim=1)
        x = self.fc2(x)
        return feat, F.log_softmax(x, dim=1)

In [5]:
train_loader2 = train_loader
test_loader2 = test_loader_A
model = Modelnsl8()
sgd = SGD(model.parameters(), lr=1e-2)
criterion = SupConLoss(temperature=0.7)
loss_fn = CrossEntropyLoss()
all_epoch = 15
batch_size = b_s

for current_epoch in range(all_epoch):
    model.train()
    for idx, (train_x, train_label) in enumerate(train_loader2):
        sgd.zero_grad()
        # 调整输入形状为 [batch_size, 1, 28, 28]
        train_x = train_x.view(-1, 1, 28, 28).float()
        features1, predict_y = model(train_x)
        
        features = torch.cat([features1, features1], dim=0)
        f1, f2 = torch.split(features, [batch_size, batch_size], dim=0)
        feat = torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1)         
        target2 = train_label
        loss2 = criterion(feat, target2)
        loss1 = loss_fn(predict_y, train_label.long())
        loss = loss1 + loss2
        if current_epoch > 10:
            sgd = SGD(model.parameters(), lr=1e-3)
        if idx % 1000 == 0:
            print('idx: {}, loss: {}'.format(idx, loss.sum().item()))
        loss.backward()
        sgd.step()
    print("epoch i=", current_epoch + 1)
    print("this train epoch end")
    all_correct_num = 0
    all_sample_num = 0
    model.eval()
    for idx, (test_x, test_label) in enumerate(test_loader2):
        # 调整输入形状为 [batch_size, 1, 28, 28]
        test_x = test_x.view(-1, 1, 28, 28).float()
        _, predict_y = model(test_x)
        predict_y = predict_y.detach()
        predict_y = np.argmax(predict_y, axis=-1)
        current_correct_num = predict_y == test_label
        all_correct_num += np.sum(current_correct_num.numpy(), axis=-1)
        all_sample_num += current_correct_num.shape[0]
    acc = all_correct_num / all_sample_num
    print('accuracy: {:.3f}'.format(acc))
print("training end")

idx: 0, loss: 45.57554626464844
idx: 1000, loss: 35.02802658081055
idx: 2000, loss: 34.66157913208008
idx: 3000, loss: 34.85332489013672
idx: 4000, loss: 34.66092300415039


KeyboardInterrupt: 